In [36]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory



# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [37]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import random

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image

from tqdm.auto import tqdm

from sklearn.manifold import TSNE
from umap import UMAP

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import resnet50

sns.set_theme(style="whitegrid")

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [38]:
DATASET_DIR = "/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset"

SIMCLR_PATH = "/kaggle/input/notebooks/mehtab9000/better-simclr/simclr_checkpoints/resnet50_backbone_epoch_100.pth"

MAE_PATH = "/kaggle/input/notebooks/mehtab9001/better-mae/mae_checkpoints/mae_encoder_epoch_120.pth"

IMAGE_DIR = os.path.join(DATASET_DIR, "valid/images")
LABEL_DIR = os.path.join(DATASET_DIR, "valid/labels")

In [39]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

class EvalDataset(Dataset):

    def __init__(
        self,
        image_dir,
        label_dir
    ):

        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)

        self.images = sorted(
            list(
                self.image_dir.glob("*")
            )
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = self.images[idx]
    
        try:
    
            image = Image.open(
                img_path
            ).convert("RGB")
    
            image = transform(image)
    
            label_path = (
                self.label_dir /
                f"{img_path.stem}.txt"
            )
    
            label = 0
    
            if label_path.exists():
    
                with open(label_path) as f:
    
                    line = f.readline().strip()
    
                    if line:
                        label = int(
                            line.split()[0]
                        )
    
            return image, label, str(img_path)
    
        except Exception as e:
    
            print(
                f"Error loading {img_path}: {e}"
            )
    
            image = torch.zeros(
                3,
                224,
                224
            )
    
            return image, 0, str(img_path)


dataset = EvalDataset(
    IMAGE_DIR,
    LABEL_DIR
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(len(dataset))

3511


In [40]:
class MAE(nn.Module):

    def __init__(
        self,
        config
    ):

        super().__init__()

        self.patch_size = config.patch_size

        self.embed_dim = config.embed_dim

        self.mask_ratio = config.mask_ratio

        self.num_patches = (
            config.image_size //
            config.patch_size
        ) ** 2

        patch_dim = (
            3 *
            config.patch_size *
            config.patch_size
        )

        self.patch_embed = nn.Conv2d(
            3,
            config.embed_dim,
            kernel_size=config.patch_size,
            stride=config.patch_size
        )

        self.pos_embed = nn.Parameter(
            torch.randn(
                1,
                self.num_patches,
                config.embed_dim
            )
        )

        self.encoder = nn.ModuleList([
            TransformerBlock(
                config.embed_dim,
                config.num_heads
            )
            for _ in range(
                config.depth
            )
        ])

        self.encoder_norm = nn.LayerNorm(
            config.embed_dim
        )

        self.decoder_embed = nn.Linear(
            config.embed_dim,
            config.decoder_dim
        )

        self.mask_token = nn.Parameter(
            torch.randn(
                1,
                1,
                config.decoder_dim
            )
        )

        self.decoder_pos_embed = nn.Parameter(
            torch.randn(
                1,
                self.num_patches,
                config.decoder_dim
            )
        )

        self.decoder = nn.ModuleList([
            TransformerBlock(
                config.decoder_dim,
                config.decoder_heads
            )
            for _ in range(
                config.decoder_depth
            )
        ])

        self.decoder_norm = nn.LayerNorm(
            config.decoder_dim
        )

        self.decoder_pred = nn.Linear(
            config.decoder_dim,
            patch_dim
        )

    def patchify(
        self,
        imgs
    ):

        p = self.patch_size

        h = w = imgs.shape[2] // p

        x = imgs.reshape(
            imgs.shape[0],
            3,
            h,
            p,
            w,
            p
        )

        x = torch.einsum(
            "nchpwq->nhwpqc",
            x
        )

        x = x.reshape(
            imgs.shape[0],
            h * w,
            p * p * 3
        )

        return x

    def unpatchify(
        self,
        x
    ):

        p = self.patch_size

        h = w = int(
            x.shape[1] ** 0.5
        )

        x = x.reshape(
            x.shape[0],
            h,
            w,
            p,
            p,
            3
        )

        x = torch.einsum(
            "nhwpqc->nchpwq",
            x
        )

        imgs = x.reshape(
            x.shape[0],
            3,
            h * p,
            h * p
        )

        return imgs

    def random_masking(
        self,
        x
    ):

        N, L, D = x.shape

        len_keep = int(
            L * (1 - self.mask_ratio)
        )

        noise = torch.rand(
            N,
            L,
            device=x.device
        )

        ids_shuffle = torch.argsort(
            noise,
            dim=1
        )

        ids_restore = torch.argsort(
            ids_shuffle,
            dim=1
        )

        ids_keep = ids_shuffle[
            :,
            :len_keep
        ]

        x_masked = torch.gather(
            x,
            dim=1,
            index=ids_keep.unsqueeze(-1).repeat(
                1,
                1,
                D
            )
        )

        mask = torch.ones(
            N,
            L,
            device=x.device
        )

        mask[:, :len_keep] = 0

        mask = torch.gather(
            mask,
            dim=1,
            index=ids_restore
        )

        return (
            x_masked,
            mask,
            ids_restore
        )

    def forward_encoder(
        self,
        imgs
    ):

        x = self.patch_embed(
            imgs
        )

        x = x.flatten(2)

        x = x.transpose(1, 2)

        x = x + self.pos_embed

        x, mask, ids_restore = self.random_masking(
            x
        )

        for block in self.encoder:

            x = block(x)

        x = self.encoder_norm(
            x
        )

        return (
            x,
            mask,
            ids_restore
        )

    def forward_decoder(
        self,
        x,
        ids_restore
    ):

        x = self.decoder_embed(
            x
        )

        B, L, D = x.shape

        mask_tokens = self.mask_token.repeat(
            B,
            ids_restore.shape[1] - L,
            1
        )

        x_ = torch.cat(
            [
                x,
                mask_tokens
            ],
            dim=1
        )

        x_ = torch.gather(
            x_,
            dim=1,
            index=ids_restore.unsqueeze(-1).repeat(
                1,
                1,
                D
            )
        )

        x_ = x_ + self.decoder_pos_embed

        for block in self.decoder:

            x_ = block(x_)

        x_ = self.decoder_norm(
            x_
        )

        pred = self.decoder_pred(
            x_
        )

        return pred


    def extract_features(self, imgs):

        x = self.patch_embed(imgs)
    
        x = x.flatten(2)
    
        x = x.transpose(1,2)
    
        x = x + self.pos_embed
    
        for block in self.encoder:
            x = block(x)
    
        x = self.encoder_norm(x)
    
        return x.mean(dim=1)

    def forward(
        self,
        imgs
    ):

        latent, mask, ids_restore = self.forward_encoder(
            imgs
        )

        pred = self.forward_decoder(
            latent,
            ids_restore
        )

        return pred, mask

In [41]:
class SimCLR(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        backbone = resnet50(
            weights=None
        )

        feature_dim = (
            backbone.fc.in_features
        )

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.projector = nn.Sequential(
            nn.Linear(
                feature_dim,
                feature_dim,
                bias=False
            ),
            nn.BatchNorm1d(
                feature_dim
            ),
            nn.ReLU(inplace=True),
            nn.Linear(
                feature_dim,
                embedding_dim,
                bias=False
            )
        )

    def forward(
        self,
        x,
        return_features=False
    ):

        features = self.backbone(x)

        projections = self.projector(
            features
        )

        projections = F.normalize(
            projections,
            dim=1
        )

        if return_features:
            return (
                features,
                projections
            )

        return projections

In [42]:
class MLP(nn.Module):

    def __init__(
        self,
        dim,
        mlp_ratio=4.0
    ):

        super().__init__()

        hidden_dim = int(
            dim * mlp_ratio
        )

        self.net = nn.Sequential(
            nn.Linear(
                dim,
                hidden_dim
            ),
            nn.GELU(),
            nn.Linear(
                hidden_dim,
                dim
            )
        )

    def forward(
        self,
        x
    ):

        return self.net(x)


class TransformerBlock(nn.Module):

    def __init__(
        self,
        dim,
        num_heads
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(
            dim
        )

        self.attn = nn.MultiheadAttention(
            dim,
            num_heads,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(
            dim
        )

        self.mlp = MLP(dim)

    def forward(
        self,
        x
    ):

        y = self.norm1(x)

        y, _ = self.attn(
            y,
            y,
            y
        )

        x = x + y

        x = x + self.mlp(
            self.norm2(x)
        )

        return x

In [43]:
class Config:
    dataset_dir = DATASET_DIR
    output_dir = "/kaggle/working/mae_checkpoints"

    image_size = 224
    patch_size = 16

    batch_size = 64

    epochs = 150

    learning_rate = 1.5e-4

    mask_ratio = 0.75

    embed_dim = 192
    depth = 12
    num_heads = 3

    decoder_dim = 256
    decoder_depth = 4
    decoder_heads = 8

    num_workers = 2

    seed = 42

config = Config()

os.makedirs(
    config.output_dir,
    exist_ok=True
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

seed_everything(
    config.seed
)

In [44]:
simclr = SimCLR(embedding_dim=128)

simclr_ckpt = torch.load(
    SIMCLR_PATH,
    map_location=device
)

simclr.backbone.load_state_dict(
    simclr_ckpt
)

simclr = simclr.to(device)
simclr.eval()

mae = MAE(config)

mae_ckpt = torch.load(
    MAE_PATH,
    map_location=device
)

mae.patch_embed.load_state_dict(
    mae_ckpt["patch_embed"]
)

mae.encoder.load_state_dict(
    mae_ckpt["encoder"]
)

mae.encoder_norm.load_state_dict(
    mae_ckpt["encoder_norm"]
)

mae.pos_embed.data.copy_(
    mae_ckpt["pos_embed"]
)

mae = mae.to(device)
mae.eval()

print("models loaded")

models loaded


In [45]:
simclr_embeddings = []
mae_embeddings = []

labels = []
paths = []

with torch.no_grad():

    for images, batch_labels, batch_paths in tqdm(loader):

        images = images.to(device)

        # SimCLR
        simclr_feat = simclr.backbone(
            images
        )

        simclr_embeddings.append(
            simclr_feat.cpu().numpy()
        )

        # MAE
        mae_feat = mae.extract_features(images)

        mae_embeddings.append(
            mae_feat.cpu().numpy()
        )

        labels.extend(
            batch_labels.numpy()
        )

        paths.extend(
            batch_paths
        )

simclr_embeddings = np.concatenate(
    simclr_embeddings,
    axis=0
)

mae_embeddings = np.concatenate(
    mae_embeddings,
    axis=0
)

labels = np.array(labels)

print(simclr_embeddings.shape)
print(mae_embeddings.shape)
print(labels.shape)

  0%|          | 0/28 [00:00<?, ?it/s]

(3511, 2048)
(3511, 192)
(3511,)


In [47]:
np.savez(
    "/kaggle/working/embeddings.npz",

    simclr=simclr_embeddings,
    mae=mae_embeddings,

    labels=labels,
    paths=np.array(paths)
)

print("saved")

saved


In [48]:
data = np.load(
    "/kaggle/working/embeddings.npz",
    allow_pickle=True
)

print(data.files)

['simclr', 'mae', 'labels', 'paths']


In [49]:
data = np.load(
    "/kaggle/working/embeddings.npz",
    allow_pickle=True
)

for k in data.files:
    print(k, data[k].shape)

simclr (3511, 2048)
mae (3511, 192)
labels (3511,)
paths (3511,)
